<a href="https://colab.research.google.com/github/ulugbek-lab/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print("Menu detector!")

Menu detector!


In [12]:
# ------------------------------
# Import Libraries
# ------------------------------
from google.colab import drive
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

from torchvision.models import mobilenet_v2

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print(DATASET_PATH)



CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot dog': 'hot-dog',
    'chocolate_cake': 'dessert',  # label grouping | class consolidation(such as dessert)
    'cheesecake': 'dessert',      # label grouping | class consolidation(such as dessert)
    'kebab': 'kebab',
    'sushi': 'sushi'
}




CLASSES = ['hamburger', 'hot-dog', 'dessert', 'kebab', 'sushi']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)} #assign number to classes
NUM_CLASSES = len(CLASSES)
print(CLASS_TO_IDX)
print(NUM_CLASSES) # 5 categories/classes


transform = transforms.Compose([
    transforms.Resize((224,224)), # resize image width & hight
    transforms.ToTensor(), # convert image to tensor; # H, W, C  => C, H, W(tensor transform)
    transforms.Normalize(mean=[0.485, 0.456,0.406],# pixel = (pixel - mean) /std
                         std=[0.229, 0.224, 0.225])
])




/content/drive/MyDrive/food101_dataset
{'hamburger': 0, 'hot-dog': 1, 'dessert': 2, 'kebab': 3, 'sushi': 4}
5


In [6]:


# ------------------------------
# Custom Dataset class
# ------------------------------


#( Step 1). Define only
class FoodDataset(Dataset):
  # (Step 4). Initialize the methods
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    print('images_length', len(self.images))
    return len(self.images)
  # (Step 6) Actual image is called
  def __getitem__(self, idx):
    img_path = self.images[idx]
    print('img_path',img_path)
    label = self.labels[idx]
    print('label',label)
  # (Step 7) Try to open the image!
    try:
      image = Image.open(img_path).convert('RGB')
    except (UnidentifiedImageError, OSError):
      print(f"Skipping broken image: {img_path}")
      return self.__getitem__((idx + 1) % len(self.images))# skip error image path
    if self.transform:
      image = self.transform(image)
  # ( Step 8) return
    return image, label
#


In [7]:
# ------------------------------
# Gather and Split class
# ------------------------------

# (Step 2). Start Mapping
all_images = [] # to store tuples
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
   class_path = os.path.join(DATASET_PATH, original_class) #/content/drive/MyDrive/food101_dataset/hamburger
   print('Class_path:', class_path)
   if not os.path.exists(class_path):
     print(f"Warning: {class_path} not found")
     continue
   for img in os.listdir(class_path):
     if img.endswith(('.jpg', '.jpeg', '.png')):
      full_path = os.path.join(class_path, img)#/content/drive/MyDrive/food101_dataset/hamburger.jpeg
      all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images) # mix all the images
split = int(0.8 * len(all_images))
train_data = all_images[:split] # 1000 | 800 train_data | 200 val_data
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)


# print('all_images', all_images)

#(Step 3). Call the class methods
dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
# (Step 5). Ask for an actual image
img, lbl = dataset[7]






Class_path: /content/drive/MyDrive/food101_dataset/hamburger
Class_path: /content/drive/MyDrive/food101_dataset/hot dog
Class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
Class_path: /content/drive/MyDrive/food101_dataset/cheesecake
Class_path: /content/drive/MyDrive/food101_dataset/kebab
Class_path: /content/drive/MyDrive/food101_dataset/sushi
images_length 1920
1920
img_path /content/drive/MyDrive/food101_dataset/sushi/Image_16.jpg
label 4


In [8]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)  # 32 images right away sent; shuffle => mix; 2 thread loading in background for speed
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

images_length 1920
images_length 1920


In [10]:
 # Pretrained model

 model = mobilenet_v2(weights="IMAGENET1K_V1") # lighweight and CNN and pre trained model; 1000 categories; (Convolutional Neural Network)
 model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone(model brain) | model layer freeze

 # ********    Pre-trained model => Fine-tuning(Training Data) => Fine-tuned Model    **************#

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 90.6MB/s]


In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
model = model.to(device)

device: cuda


In [14]:
criterion = nn.CrossEntropyLoss()# Loss Function | "70%" burger, "30%" sushi
optimizer = optim.Adam(model.parameters(), lr=0.001) # weight
test = torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10-20% speed up training


criterion CrossEntropyLoss()
optimizer Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
test True
